In [1]:
!pip install torch-geometric scikit-learn matplotlib seaborn pandas numpy

import os
import gc
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear
from torch.utils.data import DataLoader, TensorDataset

# Graph Models
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv, GCNConv
from torch_geometric.utils import k_hop_subgraph, to_undirected

# Tabular Models (sklearn)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import kneighbors_graph

# Metrics & Plotting
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, f1_score,
                             accuracy_score, precision_score, recall_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION
# ============================================================
CONFIG = {
    'data_path': '/kaggle/input/datasets/vagifa/ethereum-frauddetection-dataset/transaction_dataset.csv',
    'save_dir': '/kaggle/working/comparisons/',

    'in_channels':      None,   
    'hidden_channels':  16,        
    'out_channels':     1,

    'epochs':           5,        
    'learning_rate':    0.005,
    'weight_decay':     5e-4,      

    'batch_size':    512,
    'num_hops':      2,
    'knn_k':         5,           # Graph construction neighbors

    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)


# ============================================================
# 2. DATA PREPROCESSING & DYNAMIC GRAPH (LEAK-FREE)
# ============================================================
def load_ethereum_data(config):
    print("Loading Ethereum dataset...")
    df = pd.read_csv(config['data_path'])
    df.columns = df.columns.str.strip()
    
    # Drop non-predictive categorical columns
    cols_to_drop = ['Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
    actual_drop = [c for c in cols_to_drop if c in df.columns]
    
    y = df['FLAG'].values
    X_df = df.drop(columns=actual_drop + ['FLAG'])
    X_df = X_df.apply(pd.to_numeric, errors='coerce')
    
    n_nodes = len(X_df)
    
    # --- 1. SPLIT DATA FIRST TO PREVENT LEAKAGE ---
    indices = np.arange(n_nodes)
    np.random.seed(42)
    np.random.shuffle(indices)
    
    train_end = int(0.8 * n_nodes)
    train_idx = indices[:train_end]
    test_idx  = indices[train_end:]
    
    train_mask = torch.zeros(n_nodes, dtype=torch.bool)
    test_mask  = torch.zeros(n_nodes, dtype=torch.bool)
    
    train_mask[train_idx] = True
    test_mask[test_idx]   = True

    # --- 2. FIT PREPROCESSORS STRICTLY ON TRAIN DATA ---
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_df.iloc[train_idx])  # Fit ONLY on Train
    X_imputed = imputer.transform(X_df) # Transform all
    
    scaler = StandardScaler()
    scaler.fit(X_imputed[train_idx])   # Fit ONLY on Train
    X_scaled = scaler.transform(X_imputed) # Transform all
    
    # --- 3. BUILD GRAPH ---
    print(f"Building k-NN graph (k={config['knn_k']}) for GNN baselines...")
    A = kneighbors_graph(X_scaled, config['knn_k'], mode='connectivity', include_self=False, n_jobs=-1)
    coo = A.tocoo()
    edge_index = torch.tensor(np.vstack((coo.row, coo.col)), dtype=torch.long)
    edge_index = to_undirected(edge_index)
    
    x = torch.tensor(X_scaled, dtype=torch.float)
    y = torch.tensor(y, dtype=torch.long)
    
    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = train_mask
    data.test_mask  = test_mask
    
    config['in_channels'] = x.shape[1]
    
    print(f"Data Loaded: {data.num_nodes} nodes, {data.num_edges} edges.")
    print(f"Splits - Train: {train_mask.sum()}, Test: {test_mask.sum()}")
    return data


# ============================================================
# 3. MODELS (GNN, TRANSFORMER, CNN, DNN)
# ============================================================
# --- GNN Baseline ---
class BaselineGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, gnn_type='GCN'):
        super().__init__()
        if gnn_type == 'GCN':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels)
        elif gnn_type == 'SAGE':
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        elif gnn_type == 'GAT':
            self.conv1 = GATConv(in_channels, hidden_channels, heads=1)
            self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1)
            
        self.classifier = Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.classifier(x)

# --- Deep Neural Network (DNN) ---
class TabularDNN(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# --- 1D Convolutional Neural Network (CNN) ---
class TabularCNN(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(4) # Enforce fixed output size of 4 per channel
        self.fc = nn.Sequential(
            nn.Linear(32 * 4, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1) # (Batch, Channels, Features) -> (B, 1, F)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1) # Flatten
        return self.fc(x)

# --- Tabular Transformer ---
class TabularTransformer(nn.Module):
    def __init__(self, in_features, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Linear(1, d_model) # Treat each feature value as a "token"
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = x.unsqueeze(-1) # (B, F, 1)
        x = self.embedding(x) # (B, F, d_model)
        x = self.transformer(x) # (B, F, d_model)
        x = x.mean(dim=1) # Global Average Pooling over all features: (B, d_model)
        return self.fc(x)


# ============================================================
# 4. TRAINING & EVALUATION ROUTINES
# ============================================================
class SubgraphBatchSampler:
    def __init__(self, node_indices, batch_size, num_hops, edge_index, num_nodes, shuffle=True):
        self.node_indices = node_indices
        self.batch_size   = batch_size
        self.num_hops     = num_hops
        self.edge_index   = edge_index
        self.num_nodes    = num_nodes
        self.shuffle      = shuffle

    def __iter__(self):
        idx = self.node_indices.clone()
        if self.shuffle:
            idx = idx[torch.randperm(len(idx))]

        for start in range(0, len(idx), self.batch_size):
            seeds = idx[start: start + self.batch_size]
            subset, sub_edge_index, mapping, _ = k_hop_subgraph(
                node_idx=seeds, num_hops=self.num_hops, edge_index=self.edge_index,
                relabel_nodes=True, num_nodes=self.num_nodes, flow='source_to_target'
            )
            yield subset, sub_edge_index, mapping

    def __len__(self):
        return (len(self.node_indices) + self.batch_size - 1) // self.batch_size

def train_graph_model(model, data, config, model_name="Model"):
    print(f"\n--- Training Graph Model: {model_name} ---")
    train_idx = torch.where(data.train_mask)[0]
    test_idx  = torch.where(data.test_mask)[0]

    train_sampler = SubgraphBatchSampler(train_idx, config['batch_size'], config['num_hops'], data.edge_index, data.num_nodes, shuffle=True)
    test_sampler  = SubgraphBatchSampler(test_idx, config['batch_size'] * 2, config['num_hops'], data.edge_index, data.num_nodes, shuffle=False)

    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        for subset, sub_ei, mapping in train_sampler:
            optimizer.zero_grad()
            out = model(data.x[subset].to(config['device']), sub_ei.to(config['device'])).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            out_seed = out[mapping]
            y_seed = data.y[subset][mapping].float().to(config['device'])
            
            loss = criterion(out_seed, y_seed)
            loss.backward()
            optimizer.step()
    
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for subset, sub_ei, mapping in test_sampler:
            out = model(data.x[subset].to(config['device']), sub_ei.to(config['device'])).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            out_seed = out[mapping]
            y_seed = data.y[subset][mapping].float().to(config['device'])
            
            all_logits.append(out_seed.cpu())
            all_labels.append(y_seed.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs > 0.5).astype(int)
    
    return labels, preds, probs

def train_dl_model(model, X_train, y_train, X_test, y_test, config, model_name="DL Model"):
    print(f"\n--- Training DL Model: {model_name} ---")
    
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float), torch.tensor(y_train, dtype=torch.float))
    test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float), torch.tensor(y_test, dtype=torch.float))
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=config['batch_size']*2, shuffle=False)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    
    model.to(config['device'])
    
    for epoch in range(1, config['epochs'] + 1):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(config['device']), y_b.to(config['device'])
            optimizer.zero_grad()
            out = model(X_b).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            
    model.eval()
    all_logits = []
    with torch.no_grad():
        for X_b, _ in test_loader:
            X_b = X_b.to(config['device'])
            out = model(X_b).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            all_logits.append(out.cpu())
            
    logits = torch.cat(all_logits)
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs > 0.5).astype(int)
    
    return y_test, preds, probs


def run_tabular_baselines(data):
    print("\n--- Training Tabular ML Baselines ---")
    train_idx = data.train_mask.nonzero(as_tuple=True)[0]
    test_idx  = data.test_mask.nonzero(as_tuple=True)[0]

    X_train = data.x[train_idx].numpy()
    y_train = data.y[train_idx].numpy()
    X_test  = data.x[test_idx].numpy()
    y_test  = data.y[test_idx].numpy()

    ml_models = {
        'LR': LogisticRegression(max_iter=100, solver='liblinear'),
        'SVM': SVC(probability=True, max_iter=250),
        'MLP': MLPClassifier(hidden_layer_sizes=(16,), max_iter=50, random_state=42)
    }

    tabular_results = {}
    for name, clf in ml_models.items():
        print(f"Training {name}...")
        clf.fit(X_train, y_train)
        
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba(X_test)[:, 1]
        else:
            probs = clf.decision_function(X_test)
            
        preds = clf.predict(X_test)
        tabular_results[name] = (y_test, preds, probs)
        
    return tabular_results


# ============================================================
# 5. COMPARISON PIPELINE & PLOTTING
# ============================================================
def plot_combined_roc(results, config):
    plt.figure(figsize=(12, 10))
    colors = sns.color_palette("tab10", len(results))
    
    for (name, (y_true, preds, probs)), color in zip(results.items(), colors):
        fpr, tpr, _ = roc_curve(y_true, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, lw=2, 
                 label=f'{name} (AUC = {roc_auc:.4f})')

    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
    plt.title('Combined ROC Curve Comparison', fontsize=16, weight='bold')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '1_Combined_ROC.png'), dpi=300)
    plt.close()


def plot_combined_pr(results, config):
    plt.figure(figsize=(12, 10))
    colors = sns.color_palette("tab10", len(results))
    
    for (name, (y_true, preds, probs)), color in zip(results.items(), colors):
        precision, recall, _ = precision_recall_curve(y_true, probs)
        pr_auc = auc(recall, precision)
        plt.plot(recall, precision, color=color, lw=2, 
                 label=f'{name} (PR AUC = {pr_auc:.4f})')

    plt.title('Combined Precision-Recall Curve Comparison', fontsize=16, weight='bold')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '2_Combined_PR.png'), dpi=300)
    plt.close()


def plot_confusion_matrices_grid(results, config):
    num_models = len(results)
    cols = 3
    rows = math.ceil(num_models / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4.5))
    axes = axes.flatten()

    for idx, (name, (y_true, preds, probs)) in enumerate(results.items()):
        cm = confusion_matrix(y_true, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    annot_kws={"size": 12}, ax=axes[idx])
        axes[idx].set_title(name, fontsize=12, weight='bold')
        axes[idx].set_xticklabels(['Valid (0)', 'Fraud (1)'])
        axes[idx].set_yticklabels(['Valid (0)', 'Fraud (1)'])
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

    for empty_idx in range(num_models, len(axes)):
        fig.delaxes(axes[empty_idx])

    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '3_Confusion_Matrices_Grid.png'), dpi=300)
    plt.close()


def generate_comparison_table_and_reports(results, config):
    print("\n" + "="*70)
    print("CLASSIFICATION REPORTS (4 DIGITS) FOR ALL MODELS")
    print("="*70)
    
    metrics_data = []
    
    for name, (y_true, preds, probs) in results.items():
        print(f"\n[{name}] Classification Report:")
        print(classification_report(y_true, preds, digits=4, target_names=['Valid (0)', 'Fraud (1)'], zero_division=0))
        
        acc = accuracy_score(y_true, preds)
        mac_p = precision_score(y_true, preds, average='macro', zero_division=0)
        mac_r = recall_score(y_true, preds, average='macro', zero_division=0)
        mac_f1 = f1_score(y_true, preds, average='macro', zero_division=0)
        
        metrics_data.append([name, acc, mac_p, mac_r, mac_f1])
        
    df_metrics = pd.DataFrame(metrics_data, columns=['Model', 'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1'])
    
    print("\n" + "="*80)
    print("SUMMARY METRICS TABLE (MACRO AVERAGES)")
    print("="*80)
    print(df_metrics.to_string(index=False, float_format="{:.6f}".format))
    
    df_metrics.to_csv(os.path.join(config['save_dir'], '4_Comparison_Metrics.csv'), index=False)
    print(f"\nAll comparison outputs saved to: {config['save_dir']}")


# ============================================================
# 6. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    
    # 1. Load Data
    data = load_ethereum_data(CONFIG)
    in_features = CONFIG['in_channels']
    
    # Split data to arrays for DL & ML models
    train_idx = data.train_mask.nonzero(as_tuple=True)[0]
    test_idx  = data.test_mask.nonzero(as_tuple=True)[0]
    
    X_train, y_train = data.x[train_idx].numpy(), data.y[train_idx].numpy()
    X_test, y_test   = data.x[test_idx].numpy(), data.y[test_idx].numpy()

    all_results = {}
    
    # 2. Run Tabular Baselines (LR, SVM, MLP)
    tabular_results = run_tabular_baselines(data)
    all_results.update(tabular_results)
    
    # 3. Run PyTorch Deep Learning & Transformer Models
    dl_models = {
        'Deep Neural Net (DNN)': TabularDNN(in_features),
        '1D CNN': TabularCNN(in_features),
        'Transformer': TabularTransformer(in_features)
    }
    
    for dl_name, dl_model in dl_models.items():
        y_t, preds, probs = train_dl_model(dl_model, X_train, y_train, X_test, y_test, CONFIG, model_name=dl_name)
        all_results[dl_name] = (y_t, preds, probs)
    
    # 4. Run Graph Baselines (GCN, SAGE, GAT)
    graph_baselines = ['GCN', 'SAGE', 'GAT']
    for gnn_type in graph_baselines:
        model = BaselineGNN(in_features, CONFIG['hidden_channels'], CONFIG['out_channels'], gnn_type=gnn_type).to(CONFIG['device'])
        
        display_name = "GNN (GraphSAGE)" if gnn_type == 'SAGE' else f"GNN ({gnn_type})"
        y_true, preds, probs = train_graph_model(model, data, CONFIG, model_name=display_name)
        all_results[display_name] = (y_true, preds, probs)

    # 5. Generate Comparisons
    print("\nGenerating final comparison charts...")
    plot_combined_roc(all_results, CONFIG)
    plot_combined_pr(all_results, CONFIG)
    plot_confusion_matrices_grid(all_results, CONFIG)
    generate_comparison_table_and_reports(all_results, CONFIG)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.9 MB/s eta 0:00:00
Loading Ethereum dataset...
Building k-NN graph (k=5) for GNN baselines...
Data Loaded: 9841 nodes, 67840 edges.
Splits - Train: 7872, Test: 1969

--- Training Tabular ML Baselines ---
Training LR...
Training SVM...
Training MLP...

--- Training DL Model: Deep Neural Net (DNN) ---

--- Training DL Model: 1D CNN ---

--- Training DL Model: Transformer ---

--- Training Graph Model: GNN (GCN) ---

--- Training Graph Model: GNN (GraphSAGE) ---

--- Training Graph Model: GNN (GAT) ---

Generating final comparison charts...

CLASSIFICATION REPORTS (4 DIGITS) FOR ALL MODELS

[LR] Classification Report:
              precision    recall  f1-score   support

   Valid (0)     0.9974    0.9942    0.9958      1540
   Fraud (1)     0.9793    0.9907    0.9849       429

    accuracy                         0.9934      1969
   macro avg     0.98